# Иерархическая кластеризация и дендрограммы

> Модуль **Unsupervised Learning**, ноутбук 5. Разбираем подробно, с примерами и интуицией.
> Здесь кластеры образуют не плоское разбиение, а **дерево вложенности**, которое можно
> «разрезать» на любой высоте и получить нужное число групп.

### План занятия
1. **Когда выбирают иерархическую кластеризацию** — биология, генетика, тексты и не только.
2. **Агломеративный алгоритм** — слияние снизу вверх.
3. **Критерии связи (linkage)** — single / complete / average / Ward.
4. **Метрики расстояния** — евклид и косинус (для текстов).
5. **Дендрограмма** — как её читать.
6. **Выбор числа кластеров** — разрез дерева, правило «наибольшего разрыва».
7. **Живой пример** — кластеризация реальных текстов (датасет 20 newsgroups): дерево тем.

В конце — раздел **вопросов и ответов**.

## 1. Когда нас подталкивают к иерархической кластеризации

Иерархическая кластеризация строит **целое дерево** вложенных групп — от «каждый объект сам по
себе» до «все вместе». Это особенно ценно, когда у данных есть **естественная вложенная
структура** или когда заранее непонятно, сколько кластеров нужно. Типичные ситуации:

- **Биологическая систематика и филогенетика.** Виды объединяются в роды, роды — в семейства,
  и так далее. «Древо жизни» — это буквально дендрограмма. Иерархия здесь — не приём, а суть
  предметной области.
- **Генетика и биоинформатика.** Анализ экспрессии генов: тепловые карты (heatmap) с
  дендрограммами по строкам и столбцам показывают группы ко-экспрессируемых генов и похожие
  образцы. Это один из самых узнаваемых графиков в биологии данных.
- **Тексты и документы.** Темы естественно вкладываются в подтемы: «спорт» → «бейсбол»,
  «хоккей»; «наука» → «космос», «медицина». Дерево отражает эту таксономию.
- **Сегментация клиентов/товаров** на разных уровнях детализации одновременно.
- **Когда число кластеров неизвестно.** Не нужно заранее задавать $k$: строим дерево один раз,
  а число групп выбираем потом, «разрезав» его на удобной высоте.
- **Небольшие и средние данные, где важна интерпретируемость.** Дерево можно показать эксперту
  и обсудить — это нагляднее, чем плоский список меток.

> **Главная идея.** В отличие от K-Means/GMM, мы получаем не одно разбиение, а **всю
> иерархию сразу**; конкретное число кластеров — это уже вопрос *где разрезать дерево*.

In [ ]:
# Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_blobs, make_moons
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import pdist

try:
    from ipywidgets import interact, FloatSlider
    HAVE_WIDGETS = True
except Exception:
    HAVE_WIDGETS = False

np.random.seed(0)
plt.rcParams.update({"figure.figsize": (7, 5), "axes.grid": True, "grid.alpha": 0.3})
PALETTE = ["#2f6f4f", "#2d5d8a", "#9b3b46", "#9a6b00", "#5b4b8a", "#3a7d7b", "#a0522d"]

X_blob, y_blob = make_blobs(n_samples=150, centers=4, cluster_std=0.7, random_state=1)
X_moons, _ = make_moons(n_samples=300, noise=0.06, random_state=0)
print("datasets ready | виджеты:", "да" if HAVE_WIDGETS else "нет")

## 2. Агломеративный алгоритм (снизу вверх)

Самый распространённый вариант — **агломеративный** (bottom-up):

1. Сначала **каждый объект — отдельный кластер** ($n$ кластеров).
2. Находим **два ближайших** кластера и **сливаем** их в один.
3. Повторяем шаг 2, пока не останется один кластер, **запоминая порядок и «высоту»
   (расстояние) каждого слияния**.

Эта запись слияний и есть дерево, которое потом рисуют как **дендрограмму**. Существует и
обратный, *дивизивный* (top-down) подход — начать с одного кластера и делить, — но на практике
он дороже и используется реже.

Ключевой вопрос шага 2: а как мерить **расстояние между кластерами** (а не между отдельными
точками)? Ответ задаёт **критерий связи (linkage)**.

## 3. Критерии связи (linkage)

Пусть есть два кластера $A$ и $B$. Расстояние между ними определяют по-разному:

| Linkage | Определение | Характер кластеров |
|---|---|---|
| **single** (одиночная) | $\min_{a\in A,\,b\in B}\rho(a,b)$ — ближайшая пара | «цепочки»; ловит вытянутые формы, но чувствительна к шуму (эффект chaining) |
| **complete** (полная) | $\max_{a\in A,\,b\in B}\rho(a,b)$ — дальняя пара | компактные кластеры схожего диаметра; чувствительна к выбросам |
| **average** (средняя) | средн. $\rho(a,b)$ по всем парам | компромисс между single и complete |
| **Ward** | слияние, минимально увеличивающее суммарную внутрикластерную дисперсию | компактные, сопоставимого размера (идейно близко к K-Means); требует евклидовой метрики |

Посмотрим, как один и тот же набор данных «видят» разные linkage:

In [ ]:
# How linkage shapes clusters: same data, four criteria
def plot_lab(ax, X, labels, title):
    for lab in np.unique(labels):
        m = labels == lab
        ax.scatter(X[m, 0], X[m, 1], s=14, color=PALETTE[lab % len(PALETTE)], alpha=0.7)
    ax.set_title(title)


fig, axes = plt.subplots(2, 4, figsize=(17, 8))
for col, method in enumerate(["single", "complete", "average", "ward"]):
    lb = AgglomerativeClustering(n_clusters=4, linkage=method).fit_predict(X_blob)
    plot_lab(axes[0, col], X_blob, lb, f"{method} — блобы")
    lm = AgglomerativeClustering(n_clusters=2, linkage=method).fit_predict(X_moons)
    plot_lab(axes[1, col], X_moons, lm, f"{method} — полумесяцы")
plt.tight_layout()
plt.show()
print("Обратите внимание: single «обтекает» полумесяцы (как density-методы), а Ward предпочитает "
      "компактные шары и режет их поперёк.")

### Метрики расстояния

Критерий связи отвечает на «как мерить между кластерами», но нужна ещё **метрика между
объектами**. По умолчанию — **евклидова**. Для **текстов и эмбеддингов** обычно берут
**косинусную** (важно направление вектора, а не длина). Важный нюанс: **Ward работает только с
евклидовой** метрикой; для косинуса используйте single/complete/average.

## 4. Дендрограмма: как её читать

**Дендрограмма** — это диаграмма слияний. По горизонтали — объекты (листья), по вертикали —
**высота слияния = расстояние, на котором два кластера объединились**. Правила чтения:

- **Чем ниже** соединены две ветви, тем **более похожи** объекты/группы (слились рано, на
  маленьком расстоянии).
- **Чем выше** перекладина, тем **более различны** объединяемые группы.
- **Горизонтальный разрез** на некоторой высоте даёт разбиение: число пересечённых вертикальных
  линий = число кластеров.

Построим дендрограмму на небольшом наборе, где листья можно прочитать поштучно:

In [ ]:
# Small, readable dendrogram
Xs, ys = make_blobs(n_samples=14, centers=3, cluster_std=0.6, random_state=2)
Z = linkage(Xs, method="ward")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 5))
a1.scatter(Xs[:, 0], Xs[:, 1], c=[PALETTE[i] for i in ys], s=60)
for i, (x, y) in enumerate(Xs):
    a1.annotate(str(i), (x, y), textcoords="offset points", xytext=(5, 4), fontsize=9)
a1.set_title("Объекты (подписаны индексами)")
dendrogram(Z, ax=a2, color_threshold=0)
a2.set_title("Дендрограмма (ward): высота = расстояние слияния")
a2.set_xlabel("индекс объекта"); a2.set_ylabel("высота слияния")
plt.show()

## 5. Как выбрать число кластеров: разрез дерева

Число кластеров выбирают **после** построения дерева, проводя горизонтальную линию (порог по
высоте) и считая, сколько ветвей она пересекает. Полезные ориентиры:

- **Правило наибольшего разрыва (gap).** Ищем самый «длинный» вертикальный промежуток между
  последовательными слияниями — резкий скачок высоты означает, что мы стали объединять уже
  *непохожие* группы. Разрез проводят в этом промежутке.
- **Содержательный разрез.** Часто высоту выбирают по смыслу: на какой детализации группы
  становятся интерпретируемыми.

Подвигайте порог разреза — слева он показан линией на дендрограмме, справа — получившиеся
кластеры:

In [ ]:
Zb = linkage(X_blob, method="ward")
z_max = Zb[:, 2].max()


def cut_tree(height=0.5 * z_max):
    labels = fcluster(Zb, t=height, criterion="distance")
    k = len(np.unique(labels))
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 5))
    dendrogram(Zb, ax=a1, color_threshold=height, no_labels=True)
    a1.axhline(height, color="#9b3b46", ls="--", lw=2)
    a1.set_title(f"Разрез на высоте {height:.1f}  →  кластеров: {k}")
    a1.set_ylabel("высота слияния")
    for lab in np.unique(labels):
        m = labels == lab
        a2.scatter(X_blob[m, 0], X_blob[m, 1], s=16, color=PALETTE[(lab - 1) % len(PALETTE)], alpha=0.7)
    a2.set_title(f"Получившиеся кластеры: {k}")
    plt.show()


if HAVE_WIDGETS:
    interact(cut_tree, height=FloatSlider(min=1.0, max=z_max * 1.05, step=0.5, value=0.5 * z_max,
                                          description="высота разреза"))
else:
    cut_tree(0.5 * z_max)

### Как это выглядит в биоинформатике: тепловая карта с дендрограммами

В анализе экспрессии генов классический график — **clustermap**: строки (гены) и столбцы
(образцы) переставляются по результатам иерархической кластеризации, а по краям рисуются
дендрограммы. Блоки одинакового цвета — это группы ко-экспрессируемых генов в группах похожих
образцов. Сгенерируем синтетические данные с тремя блоками и посмотрим, как кластеризация сама
их «соберёт»:

In [ ]:
# Synthetic gene-expression matrix with a 3x3 block structure
rng = np.random.default_rng(0)
G, S = 30, 12                                   # 30 genes, 12 samples
M = rng.normal(0, 0.5, (G, S))
for b in range(3):                              # three co-expressed blocks
    M[b * 10:(b + 1) * 10, b * 4:(b + 1) * 4] += 2.5
M_shuffled = M[rng.permutation(G)][:, rng.permutation(S)]   # scramble rows & cols

cg = sns.clustermap(M_shuffled, cmap="vlag", center=0, figsize=(7, 7),
                    cbar_pos=(0.02, 0.83, 0.03, 0.12))
cg.figure.suptitle("Clustermap: кластеризация сама восстановила блоки генов/образцов", y=1.02)
plt.show()

## 6. Живой пример: кластеризация реальных текстов (20 newsgroups)

Теперь — полноценный разбор на настоящих данных. Возьмём датасет **20 newsgroups**: это архив
сообщений из **тематических интернет-конференций** (newsgroups — предшественники форумов).
Каждое сообщение было опубликовано в одной из групп — например, `sci.space` (космос),
`rec.sport.hockey` (хоккей), `comp.graphics` (компьютерная графика). У нас тысячи таких
сообщений, и для каждого известно, **из какой оно группы**.

Возьмём девять групп из четырёх «семейств» тем (компьютеры, спорт, наука, политика/религия).
Метки (из какой группы сообщение) используем **только для проверки в конце** — сами алгоритмы их
не видят: задача кластеризации в том, чтобы **обнаружить структуру тем**, не зная ответов.
Датасет скачивается автоматически (~14 МБ при первом запуске).

In [ ]:
from sklearn.datasets import fetch_20newsgroups

cats = ["comp.graphics", "comp.sys.mac.hardware", "rec.sport.baseball", "rec.sport.hockey",
        "sci.space", "sci.med", "talk.politics.mideast", "talk.politics.guns",
        "soc.religion.christian"]
news = fetch_20newsgroups(subset="train", categories=cats,
                          remove=("headers", "footers", "quotes"), random_state=0)
y_news = np.array(news.target)
names = news.target_names
print("Всего сообщений:", len(news.data), "| тематических групп:", len(names))

# Look at one real message, so we see what we are actually clustering
print("\n--- пример сообщения из группы '%s' ---\n" % names[y_news[5]])
print(news.data[5][:400].strip(), "...")

### 6.1. Как превратить текст в числа: «мешок слов» и TF-IDF

Алгоритмы кластеризации работают с **числовыми векторами**, а у нас — текст. Значит, каждое
сообщение нужно превратить в набор чисел. Самый базовый способ — **«мешок слов» (bag of words)**:
посчитать, сколько раз каждое слово встречается в документе (порядок слов при этом игнорируется).
Тогда документ — это вектор длиной во весь словарь.

У «мешка слов» есть беда: самые частые слова (`the`, `is`, `and`) встречаются везде и забивают
сигнал, хотя про тему ничего не говорят. Это лечит **TF-IDF** — взвешивание из двух множителей:

- **TF (term frequency)** — как часто слово встречается **в данном документе**. Чем чаще, тем
  оно для этого документа важнее.
- **IDF (inverse document frequency)** — насколько слово **редкое по всей коллекции**. Если
  слово есть почти в каждом документе (`the`) — оно неинформативно, вес низкий; если редкое и
  всё же встретилось — информативно, вес высокий. Формула:
  $\;\text{IDF}(t) = \log\dfrac{N}{\text{df}(t)}$, где $N$ — число документов, $\text{df}(t)$ —
  в скольких из них встречается слово $t$.

Итог: $\;\text{TF-IDF}(t, d) = \text{TF}(t, d)\times \text{IDF}(t)$ — вес **велик**, когда слово
часто в *этом* документе, но редко в остальных, то есть когда оно **характерно** именно для
этого текста. Посмотрим на крошечном примере из трёх предложений:

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

toy = ["the cat drinks milk",
       "the cat chases the mouse",
       "the dog gnaws a bone"]
demo_vec = TfidfVectorizer()
demo_mat = demo_vec.fit_transform(toy).toarray()

# rows = documents, columns = words, values = TF-IDF weight
pd.DataFrame(demo_mat.round(2), columns=demo_vec.get_feature_names_out(),
             index=["док-1", "док-2", "док-3"])

Прочитайте таблицу: строки — документы, столбцы — слова, значения — вес TF-IDF. Слово `the` есть
во всех трёх документах → его IDF мал → вес низкий. А уникальные слова (`milk`, `bone`, `mouse`)
получают **высокий** вес: именно они отличают документы друг от друга. Так текст превращается в
осмысленные числовые векторы.

Применим то же к нашим сообщениям. Параметры `TfidfVectorizer`:
- `stop_words="english"` — выбросить служебные слова (`the`, `is`, …);
- `min_df=3` — игнорировать слова, встретившиеся реже чем в 3 документах (опечатки, мусор);
- `max_features=5000` — оставить 5000 самых полезных слов.

In [ ]:
tfidf = TfidfVectorizer(max_features=5000, stop_words="english", min_df=3)
X_tfidf = tfidf.fit_transform(news.data)            # sparse matrix: documents x words
print("Матрица TF-IDF:", X_tfidf.shape, "(документов x слов)")
print("Ненулевых значений: %.2f%% (всё остальное — нули, матрица разреженная)"
      % (100 * X_tfidf.nnz / (X_tfidf.shape[0] * X_tfidf.shape[1])))

### 6.2. Как запустить агломеративную кластеризацию (по шагам)

Тексты стали векторами — теперь кластеризация это три шага:

1. **Взять числовые векторы** (у нас — TF-IDF).
2. **Выбрать метрику и linkage** (для текстов — косинусная метрика + `average`).
3. **Запустить алгоритм** и получить метки кластеров.

В Python есть два пути, делающих одно и то же:
- **`sklearn`**: `AgglomerativeClustering(n_clusters=k, metric=..., linkage=...)` → сразу
  **метки** (удобно для дальнейшей работы);
- **`scipy`**: `linkage(...)` строит дерево, `dendrogram(...)` его рисует, `fcluster(...)`
  нарезает дерево на метки (удобно, когда нужна **дендрограмма**).

Покажем оба на прозрачном маленьком примере. Усредним TF-IDF внутри каждой темы — получится
**«усреднённое сообщение» каждой группы** (9 векторов-центров) — и сгруппируем эти 9 центров:

In [ ]:
# Step 1: numeric vectors = average TF-IDF per category ("a typical message of the topic")
centroids = np.vstack([np.asarray(X_tfidf[y_news == i].mean(0)).ravel() for i in range(len(names))])
print("Центров тем:", centroids.shape[0], "| размерность каждого:", centroids.shape[1])

# --- Way A: scikit-learn returns cluster labels directly ---
agg = AgglomerativeClustering(n_clusters=4, metric="cosine", linkage="average")
labels_c = agg.fit_predict(centroids)
print("\nsklearn — что с чем сгруппировалось (4 кластера):")
for g in sorted(set(labels_c)):
    print(f"  кластер {g}:", [names[i] for i in range(len(names)) if labels_c[i] == g])

# --- Way B: scipy builds the same tree, then we cut it into 4 clusters ---
Zc = linkage(pdist(centroids, metric="cosine"), method="average")
labels_scipy = fcluster(Zc, t=4, criterion="maxclust")
print("\nscipy fcluster даёт те же группы (метки по темам):", labels_scipy)

Результат осмысленный: алгоритм **сам** собрал компьютерные темы вместе, спортивные — вместе,
научные — вместе, а политику с религией — отдельно. Меток он при этом не использовал. Теперь
нарисуем **всё дерево** этих центров — дендрограмму тем:

In [ ]:
# Full dendrogram of the topic centroids (uses the linkage Z computed above)
fig, ax = plt.subplots(figsize=(11, 6))
dendrogram(Zc, labels=names, orientation="right", ax=ax, color_threshold=0.9)
ax.set_title("Дерево тем 20 newsgroups (центры категорий, косинус + average)")
ax.set_xlabel("высота слияния (косинусное расстояние)")
plt.tight_layout()
plt.show()

Прочитайте дерево: родственные группы сливаются **рано** (низкая высота) и стоят рядом —
`comp.graphics` с `comp.sys.mac.hardware`, `rec.sport.baseball` с `rec.sport.hockey`,
`sci.space` с `sci.med`; политика и религия — в своей ветви. Иерархия восстановила осмысленную
**таксономию тем**, хотя меток не видела.

### 6.3. Кластеризация отдельных документов (а не центров)

Сгруппировать 9 усреднённых центров легко. Куда труднее — кластеризовать **тысячи отдельных
сообщений**: вектор каждого очень разрежен (почти сплошные нули) и шумен. Стандартный приём —
сначала **сжать** TF-IDF методом **LSA** (`TruncatedSVD` — тот самый SVD из ноутбука 1,
применённый к текстам): он оставляет главные «смысловые» направления и убирает шум. Затем
кластеризуем сжатые векторы методом Ward. Качество текстовой кластеризации обычно **скромное** —
посмотрим на честную цифру и на то, какие темы путаются между собой.

In [ ]:
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize

mask = np.asarray(X_tfidf.sum(1)).ravel() > 0       # drop documents that became empty
X_lsa = normalize(TruncatedSVD(n_components=100, random_state=0).fit_transform(X_tfidf[mask]))
y_doc = y_news[mask]
print("После LSA:", X_lsa.shape, " (было слов:", X_tfidf.shape[1], ")")

labels_doc = AgglomerativeClustering(n_clusters=len(names), linkage="ward").fit_predict(X_lsa)
print("ARI документов (Ward по LSA):", round(adjusted_rand_score(y_doc, labels_doc), 3))
print("Текст кластеризуется шумно — это нормально: темы пересекаются по лексике.")

ct = pd.crosstab(pd.Series([names[i] for i in y_doc], name="тема"),
                 pd.Series(labels_doc, name="кластер"))
print("\nТема x кластер (видно, где темы совпали с кластерами, а где смешались):")
print(ct)

## 7. Плюсы, минусы и когда не стоит

**Плюсы:** не нужно заранее задавать $k$; даёт всю иерархию и наглядную дендрограмму; работает с
произвольной метрикой (через linkage); хорошо подходит для интерпретации экспертом.

**Минусы:**
- **Дорого по ресурсам.** Базовые реализации — порядка $O(n^2)$ памяти и $O(n^2\log n)$–$O(n^3)$
  времени. Для **сотен тысяч** объектов уже тяжело — там берут K-Means/HDBSCAN или сначала
  агрегируют данные.
- **Жадность и необратимость.** Слияния не пересматриваются: ранняя ошибка остаётся в дереве.
- **Чувствительность к linkage и метрике** — результат заметно зависит от выбора.

**Когда брать:** небольшие/средние данные, нужна иерархия или интерпретируемое дерево
(биология, тексты, разведочный анализ). **Когда нет:** очень большие данные или когда нужна
только быстрая плоская разметка.

## 8. Выводы

- Иерархическая кластеризация даёт **дерево вложенных групп**, а не одно разбиение; число
  кластеров выбирают потом, **разрезая** дерево.
- **Агломеративный** алгоритм сливает ближайшие кластеры снизу вверх; «ближайшие» зависит от
  **linkage** (single/complete/average/Ward) и **метрики** (евклид/косинус).
- **Дендрограмма**: высота перекладины = расстояние слияния; низко — похожи, высоко — различны;
  горизонтальный разрез задаёт число кластеров (правило наибольшего разрыва).
- На **реальных текстах** дерево по центрам категорий восстановило осмысленную **таксономию
  тем**; кластеризация отдельных документов — ожидаемо шумная (текст труден).
- Метод дорог ($\sim O(n^2)$) — для очень больших данных берут другие алгоритмы.

**Дальше:** заключительный ноутбук-капстоун — собираем весь модуль (снижение размерности +
все методы кластеризации) на эмбеддингах дикторов от начала до конца.

## Вопросы и ответы

**1) Какой linkage выбрать по умолчанию?**
Часто **Ward** (компактные кластеры, устойчив) для евклидовой метрики. Для текстов/эмбеддингов —
**average/complete с косинусной** метрикой. Single берут, когда важны вытянутые формы, но он
чувствителен к шуму (chaining).

**2) Как именно решить, где разрезать дерево?**
Ищите самый большой вертикальный «разрыв» между слияниями и режьте там; либо выбирайте высоту по
смыслу задачи (нужная детализация). Технически разрез делает `fcluster(Z, t, criterion=...)`.

**3) Почему дендрограмма по центрам категорий красивее, чем по документам?**
Усреднение по категории убирает шум отдельных сообщений и оставляет «суть темы». Поэтому центры
разделяются чисто, а отдельные документы — пересекаются по лексике и кластеризуются шумнее.

**4) Чем `AgglomerativeClustering` из sklearn отличается от `linkage` из scipy?**
Это две реализации одного подхода. `scipy.linkage` + `dendrogram` удобны для **рисования**
дерева; `sklearn.AgglomerativeClustering` — для получения **меток** в едином API с другими
алгоритмами. Часто используют обе вместе.

**5) Можно ли применять к большим данным?**
Напрямую — нет (квадратичная сложность). Варианты: сначала уменьшить число объектов (например,
K-Means на много мелких кластеров, затем иерархия над их центрами) или взять масштабируемые
методы.

**6) Hierarchical или K-Means/HDBSCAN — что лучше?**
Зависит от цели. Нужна **иерархия/дерево и интерпретация** — иерархическая. Нужны **скорость на
больших данных** — K-Means. Нужны **произвольные формы и шум** — HDBSCAN. Это инструменты под
разные задачи, а не конкуренты.